# 단일 도착지 최저가 직항 찾기 - Colab 초보자용

이 노트북은 설치가 어려운 사람도 **구글 Colab에서 버튼처럼 실행**해서 앱을 열 수 있게 만든 버전입니다.

아래 순서대로 진행하세요.

1. 상단 메뉴에서 `Runtime` -> `Run all`을 누르거나, 각 코드 칸 왼쪽의 ▶ 버튼을 위에서부터 누릅니다.
2. 설치가 끝나면 앱이 Colab 안에서 실행됩니다.
3. `앱 새 창으로 열기`가 뜨면 눌러서 검색 화면으로 들어갑니다.

> 가격은 자동으로 읽은 참고용입니다. 실제 결제 전에는 항공사, 수하물, 최종 결제 금액을 꼭 다시 확인하세요.

## 0. 처음 보는 사람용 사용 방법

- 코드 칸 왼쪽의 **▶ 버튼**을 누르면 실행됩니다.
- 실행 중에는 왼쪽 아이콘이 빙글빙글 돕니다.
- 초록 체크 또는 출력 메시지가 나오면 다음 칸으로 넘어가면 됩니다.
- 중간에 권한 확인 창이 나오면 `Run anyway` 또는 `계속 실행`을 눌러주세요.

<img src="https://raw.githubusercontent.com/jjunsss/cheapest-flights/main/docs/screenshots/01-search.png" width="760" alt="검색 입력 화면">

In [ ]:
#@title 1. 준비하기: 앱 파일과 브라우저 엔진 설치
#@markdown 처음 실행할 때는 보통 2~5분 정도 걸립니다. 다음 실행부터는 더 빨라질 수 있습니다.

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/jjunsss/cheapest-flights.git"
APP_DIR = Path("/content/cheapest-flights")

def run(command, cwd=None):
    print("$", " ".join(command) if isinstance(command, list) else command)
    subprocess.run(command, cwd=cwd, check=True)

if not APP_DIR.exists():
    run(["git", "clone", "--depth", "1", REPO_URL, str(APP_DIR)])
else:
    print("이미 앱 폴더가 있습니다. 최신 버전으로 갱신합니다.")
    run(["git", "pull", "--ff-only"], cwd=APP_DIR)

os.chdir(APP_DIR)
run(["npm", "install"], cwd=APP_DIR)
run(["npx", "playwright", "install", "chromium", "--with-deps"], cwd=APP_DIR)

print("\n준비 완료. 이제 아래의 2번 셀을 실행하세요.")

In [ ]:
#@title 2. 앱 실행하기
#@markdown 이 칸은 값을 고르지 않고 왼쪽 ▶ 버튼만 누르면 됩니다.

from pathlib import Path
import os
import signal
import socket
import subprocess
import time

APP_DIR = Path("/content/cheapest-flights")
LOG_PATH = APP_DIR / "colab-app.log"
PID_PATH = Path("/content/flight_app.pid")
SEARCH_WIDTH = 24
PARALLEL_SEARCHES = 2

def stop_previous():
    if PID_PATH.exists():
        try:
            pid = int(PID_PATH.read_text().strip())
            os.kill(pid, signal.SIGTERM)
            time.sleep(1)
        except Exception:
            pass
    subprocess.run("pkill -f 'npm run dev' || true", shell=True)
    subprocess.run("pkill -f 'tsx watch src/server/index.ts' || true", shell=True)
    subprocess.run("pkill -f 'vite --host' || true", shell=True)

def wait_for_port(port, timeout=45):
    deadline = time.time() + timeout
    while time.time() < deadline:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(1)
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(1)
    return False

stop_previous()

env = os.environ.copy()
env.update({
    "HOST": "127.0.0.1",
    "PORT": "3001",
    "FLIGHT_MAX_PAIRS": str(SEARCH_WIDTH),
    "FLIGHT_PARALLEL": str(PARALLEL_SEARCHES),
    "FLIGHT_BLOCK_HEAVY_RESOURCES": "1",
    "FLIGHT_BLOCK_IMAGES": "0",
})

log_file = open(LOG_PATH, "w", encoding="utf-8")
process = subprocess.Popen(
    ["npm", "run", "dev"],
    cwd=APP_DIR,
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)
PID_PATH.write_text(str(process.pid))

front_ready = wait_for_port(5173, 60)
back_ready = wait_for_port(3001, 60)

if front_ready and back_ready:
    print("앱 서버 준비 완료. 아래 3번 셀에서 앱을 여세요.")
else:
    print("앱이 아직 준비되지 않았습니다. 20초 기다렸다가 3번 셀을 눌러보고, 안 되면 문제 해결 셀을 확인하세요.")
    print("로그 위치:", LOG_PATH)


In [ ]:
#@title 3. 앱 열기
#@markdown 버튼이 뜨면 눌러서 앱을 엽니다. 화면이 작으면 새 창으로 여는 쪽이 더 편합니다.

try:
    from google.colab import output
    print("아래 버튼을 눌러 앱을 여세요.")
    output.serve_kernel_port_as_window(5173)
    print("\n노트북 안에서도 바로 볼 수 있게 아래에 앱을 띄웁니다.")
    output.serve_kernel_port_as_iframe(5173, height=900)
except Exception as error:
    print("Colab 포트 열기에 실패했습니다:", error)
    print("2번 셀을 다시 실행한 뒤 이 셀을 한 번 더 눌러보세요.")

## 4. 앱에서 누르는 순서

1. `출발지`를 선택합니다. 예: `ICN 서울/인천`
2. `도착지`를 선택합니다. 예: `FUK 후쿠오카`
3. `검색 시작일`, `검색 마지막일`을 고릅니다.
4. `최소 숙박일수`, `최대 숙박일수`를 입력합니다.
5. `최저가 찾기`를 누릅니다.
6. 결과가 뜨면 가격, 출국일, 숙박일수, 수하물 필터를 보면서 후보를 고릅니다.
7. 행을 누르면 상세 정보가 열립니다.

<img src="https://raw.githubusercontent.com/jjunsss/cheapest-flights/main/docs/screenshots/03-results.png" width="820" alt="결과 화면">

<img src="https://raw.githubusercontent.com/jjunsss/cheapest-flights/main/docs/screenshots/04-detail.png" width="820" alt="상세 화면">

## 5. 초보자용 팁

- 처음에는 날짜 범위를 너무 넓히지 말고 2~4주 정도로 테스트하세요.
- 결과가 적게 보이면 날짜 범위를 조금 넓혀 다시 검색하세요.
- `상세 확인 필요`라고 뜨는 항공사는 자동 추출이 불확실하다는 뜻입니다. 예약 화면에서 다시 확인하세요.
- 위탁 수하물은 사이트마다 표시 방식이 달라서, 최종 결제 전 반드시 다시 확인해야 합니다.

In [ ]:
#@title 문제 해결: 앱 로그 보기
#@markdown 앱이 안 열리거나 결과가 안 나오면 이 셀을 실행해서 마지막 로그를 확인하세요.

from pathlib import Path
LOG_PATH = Path("/content/cheapest-flights/colab-app.log")
if LOG_PATH.exists():
    print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-6000:])
else:
    print("아직 로그 파일이 없습니다. 2번 셀을 먼저 실행하세요.")

In [ ]:
#@title 사용 끝: 앱 종료하기
#@markdown 검색을 끝냈으면 이 셀을 실행해서 Colab 안의 서버를 멈춥니다.

from pathlib import Path
import os
import signal
import subprocess

PID_PATH = Path("/content/flight_app.pid")
if PID_PATH.exists():
    try:
        os.kill(int(PID_PATH.read_text().strip()), signal.SIGTERM)
    except Exception:
        pass
    PID_PATH.unlink(missing_ok=True)

subprocess.run("pkill -f 'npm run dev' || true", shell=True)
subprocess.run("pkill -f 'tsx watch src/server/index.ts' || true", shell=True)
subprocess.run("pkill -f 'vite --host' || true", shell=True)
print("앱을 종료했습니다. Colab 탭을 닫아도 됩니다.")